# Rossmann Store Sales Forecasting — Analytical Walkthrough

## Purpose and how to use this notebook

This notebook is the end-to-end **analytical guide** to the assignment. It covers every requested stage from problem framing through model selection and points to the detailed implementation, findings, charts and experiment evidence already organized in this repository.

**Avoiding duplication:** reusable Python implementation lives in `src/`; detailed findings and charts live in `reports/`; experiment artifacts live in `models/experiments/` and `data/processed/`. The notebook summarizes the decisions and results and provides a review path to those authoritative files.

Open `README.md` for the complete file-by-file map, environment setup and reproduction instructions. Raw competition data is not committed.

## 1. Problem framing and success criteria

Forecast daily sales for 1,115 stores over a 42-day horizon. The forecast should be evaluated chronologically and recursively, reflecting the fact that future actual sales are not available when later forecast dates are generated.

Success is assessed using MAE, RMSE, WMAPE and RMSLE, alongside store- and segment-level error, directional bias, explainability and operational suitability. The objective is a reliable planning input—not a claim of measured commercial uplift.

**Evidence:** `reports/model_evaluation_report.md` (problem framing and evaluation design).

## 2. Data audit and integrity

The training data contains 1,017,209 rows across 1,115 stores, from 1 January 2013 to 31 July 2015. The audit identified 844,392 open store-days, 172,817 closed store-days, 54 open-store zero-sales rows, no negative sales and no weekday/date mismatches.

**Implementation:** `src/preprocessing/data_validation.py`, `src/preprocessing/data_investigation.py`, `src/preprocessing/prepare_data.py`.

**Evidence:** `reports/data_findings.md`, `reports/data_audit.xlsx`, `reports/data_investigation.xlsx`, `reports/data_investigation_details.xlsx`.

## 3. Exploratory data analysis

EDA examined sales trends and distributions, weekday/month patterns, store types and assortment, daily and extended promotions, holidays, customers and competition. Daily Promo observations had higher historical mean sales than non-promotion observations; this is descriptive association, not causal impact. Sunday observations are sparse and not representative of the full store network.

**Implementation:** `src/preprocessing/eda.py`, `src/preprocessing/customer_eda.py`.

**Findings:** `reports/eda_findings.md`, `reports/customer_eda_findings.md`, `reports/eda_summary.xlsx`, `reports/customer_eda_summary.xlsx`.

**Charts:** `reports/figures/` and `reports/customer_eda_charts/`.

## 4. Leakage assessment and data decisions

`Customers` is excluded because it is not available at prediction time in the test/future setting. `Id` is retained only for prediction mapping. Future validation sales are never used to construct lag or rolling features: the recursive process appends predictions, not actual future values, to the history. Closed stores receive zero forecasts; model fitting uses open-store observations. Data-quality exceptions and competition placeholders are treated explicitly.

**Implementation and checks:** `src/features/recursive_forecasting.py`, `src/features/validate_recursive_features.py`, `src/models/recursive_forecaster.py`.

**Decision rationale:** `reports/model_evaluation_report.md` (data audit, leakage controls and feature engineering).

## 5. Feature engineering

The configured candidate set contains 34 features; training-only constant-feature removal selected 33 features for the final fitted model. Feature groups cover store and calendar attributes, promotions and holidays, competition, and historical sales (lags, rolling means, store mean and store-weekday mean). The selected feature schema is fixed for inference.

**Implementation:** `src/features/build_features.py`, `src/features/recursive_forecasting.py`, `src/models/feature_config.py`.

**Related diagnostics:** `reports/feature_analysis_charts/` and the feature-relationship outputs in `reports/`.

## 6. Forecasting protocol and baselines

Two chronological windows were evaluated, each covering 42 days and 46,830 store-date rows. Window 1 trains through 8 May 2015 and validates 9 May–19 June; Window 2 trains through 19 June and validates 20 June–31 July. Window 1 supports model comparison and tuning; Window 2 evaluates the frozen selected configuration on a later period.

Baselines include seasonal naive (lag 7), seven-day moving average and naive (lag 1).

**Implementation:** `src/models/baselines.py`, `src/models/run_baselines.py`, `src/models/run_first_window_baselines.py`.

**Evidence:** `reports/first_window_baseline_results.csv`.

## 7. Candidate models and hyperparameter tuning

Random Forest and LightGBM were compared. Optuna tuned LightGBM using recursive Window 1 WMAPE. The selected hyperparameters were frozen before Window 2; no further tuning was performed on that later holdout.

**Implementation:** `src/models/train_random_forest.py`, `src/models/train_lightgbm.py`, `src/models/tune_lightgbm.py`.

**Experiment evidence:** `models/experiments/lightgbm_tuning/` (best parameters, trials, run configuration, model and predictions); `models/experiments/lightgbm_second_window/`.

## 8. Model evaluation and comparison

Tuned LightGBM achieved 7.8206% WMAPE in Window 1 and 8.5643% in Window 2. In Window 2 it outperformed untuned LightGBM (8.7480%) and Random Forest (9.2387%) on WMAPE. The simple seasonal-naive baseline recorded 37.23% WMAPE in Window 1.

**Implementation:** `src/models/evaluate_tuned_lightgbm_second_window.py` and training/evaluation scripts under `src/models/`.

**Evidence:** `reports/first_window_baseline_results.csv`, `reports/second_window_tuned_lightgbm_results.json`, model result CSVs and experiment predictions.

## 9. Error analysis

Window 2 error analysis is available at store, store-type, sales-volume, promotion and calendar levels. Low-volume observations have higher proportional error than high-volume observations. Promotion-active segments and some individual stores also need additional attention. The model exhibits positive aggregate bias, so error magnitude alone is not a sufficient deployment criterion.

**Implementation:** `src/evaluation/analyze_store_performance.py`, `analyze_sales_volume_performance.py`, `analyze_promotion_performance.py`, `analyze_calendar_performance.py`.

**Evidence:** `src/evaluation/results/` (including store-, store-type-, volume-, promotion- and calendar-level CSVs).

## 10. Explainability

Native LightGBM TreeSHAP contributions were calculated on 40,282 open-store observations and 33 features. Additivity validation passed with maximum error 1.24e-10. Store-weekday historical mean, daily Promo and recent sales history were among the leading global contributors. SHAP attribution explains model behaviour; it does not establish causation.

**Implementation:** `src/evaluation/explain_lightgbm_model.py`.

**Evidence:** `src/evaluation/results/shap_global_importance.csv`, `shap_local_explanations.csv`, `shap_contribution_validation.json`.

## 11. Model selection and recommendation

**Selected candidate: tuned LightGBM.** It leads the evaluated candidates on the core forecast-error measures in both chronological windows and retains its lead in the later fixed-parameter holdout. Recommend advancing it to controlled production validation—not immediate autonomous rollout—because positive bias, low-volume performance, promotion periods and store-level exceptions require operational controls.

**Decision evidence:** `reports/model_evaluation_report.md` and `reports/executive_summary.md`.

## 12. Reviewer navigation

- Full project map and run instructions: `README.md`.
- Detailed audit and EDA findings: `reports/`.
- Executable implementation: `src/preprocessing/`, `src/features/`, `src/models/`, `src/evaluation/`.
- Charts: `reports/figures/`, `reports/customer_eda_charts/`, `reports/feature_analysis_charts/`.
- Saved experiment and validation evidence: `models/experiments/`, `data/processed/`, `src/evaluation/results/`.
- Formal evaluation report: `reports/model_evaluation_report.md`.
- Production architecture: `architecture/production_architecture.md`.

The notebook is deliberately a guided analytical narrative. The detailed source code, charts and supporting documents remain in their dedicated folders and are not duplicated here.